# EDA: Riot wiki reranker, Wikipedia Clickstream proxy

Train split only (features 2026-07, labels 2026-08). Test rows are never profiled. Built by `modeling/build_dataset.py`.

In [1]:
import polars as pl
from pathlib import Path
D = Path('../../../data/wikipedia-clickstream/modeling')
pairs = pl.read_parquet(D / 'pairs_train.parquet')
q = pl.read_parquet(D / 'queries_train.parquet')
print(pairs.shape, f'{pairs.estimated_size("mb"):.1f} MB')
pairs.head()

(6185251, 10) 206.4 MB


q,item,cand_opened_next,cand_twohop,cand_reverse,cand_popular,r,label,q_slice,item_slice
i32,i32,bool,bool,bool,bool,i64,f64,str,str
8,118200,false,false,false,true,0,0.0,"""long_tail""","""head"""
8,131817,false,false,false,true,0,0.0,"""long_tail""","""head"""
8,148027,false,false,false,true,0,0.0,"""long_tail""","""head"""
8,166788,false,false,false,true,0,0.0,"""long_tail""","""head"""
8,195798,false,false,false,true,0,0.0,"""long_tail""","""head"""


## Nulls

In [2]:
pairs.null_count()

q,item,cand_opened_next,cand_twohop,cand_reverse,cand_popular,r,label,q_slice,item_slice
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0


## Label: zero-inflated and heavy-tailed

In [3]:
pairs.select(pos_rate=(pl.col('r') > 0).mean(), label_mean=pl.col('label').mean())

pos_rate,label_mean
f64,f64
0.12952,0.480864


In [4]:
pairs.filter(pl.col('r') > 0).select([pl.col('r').quantile(p).alias(f'r_p{int(p*1000)}') for p in (0.5, 0.9, 0.99, 0.999)] + [pl.col('r').max().alias('r_max')])

r_p500,r_p900,r_p990,r_p999,r_max
f64,f64,f64,f64,i64
30.0,190.0,1483.0,7587.0,370264


In [5]:
for side in ('q_slice', 'item_slice'):
    print(pairs.group_by(side).agg(pairs=pl.len(), pos_rate=(pl.col('r') > 0).mean(), label_mean=pl.col('label').mean()).sort(side))

shape: (5, 4)
┌───────────┬─────────┬──────────┬────────────┐
│ q_slice   ┆ pairs   ┆ pos_rate ┆ label_mean │
│ ---       ┆ ---     ┆ ---      ┆ ---        │
│ str       ┆ u32     ┆ f64      ┆ f64        │
╞═══════════╪═════════╪══════════╪════════════╡
│ dormant   ┆ 216619  ┆ 0.006514 ┆ 0.01857    │
│ head      ┆ 1017465 ┆ 0.350125 ┆ 1.462835   │
│ long_tail ┆ 1825266 ┆ 0.041422 ┆ 0.122299   │
│ new       ┆ 301486  ┆ 0.092386 ┆ 0.35314    │
│ torso     ┆ 2824415 ┆ 0.12038  ┆ 0.407931   │
└───────────┴─────────┴──────────┴────────────┘


shape: (5, 4)
┌────────────┬─────────┬──────────┬────────────┐
│ item_slice ┆ pairs   ┆ pos_rate ┆ label_mean │
│ ---        ┆ ---     ┆ ---      ┆ ---        │
│ str        ┆ u32     ┆ f64      ┆ f64        │
╞════════════╪═════════╪══════════╪════════════╡
│ dormant    ┆ 104     ┆ 0.076923 ┆ 0.206547   │
│ head       ┆ 4435245 ┆ 0.096076 ┆ 0.374554   │
│ long_tail  ┆ 344787  ┆ 0.230336 ┆ 0.714122   │
│ new        ┆ 12127   ┆ 0.175229 ┆ 0.676393   │
│ torso      ┆ 1392988 ┆ 0.210657 ┆ 0.759938   │
└────────────┴─────────┴──────────┴────────────┘


## Queries and candidates per query

In [6]:
q.group_by('q_slice', 'age').agg(n=pl.len(), views_f_median=pl.col('views_f').median(), src_out_f_median=pl.col('src_out_f').median()).sort('q_slice', 'age')

q_slice,age,n,views_f_median,src_out_f_median
str,str,u32,f64,f64
"""dormant""","""established""",9812,0.0,0.0
"""dormant""","""unknown""",188,0.0,0.0
"""head""","""established""",9948,8432.5,3172.5
"""head""","""unknown""",52,6822.0,3138.0
"""long_tail""","""established""",50421,212.0,23.0
"""long_tail""","""unknown""",269,188.0,23.0
"""new""","""born_after_T""",6219,0.0,0.0
"""new""","""new_lt14d""",2033,110.0,0.0
"""torso""","""established""",38422,1022.0,247.0


In [7]:
pairs.group_by('q').agg(n_cand=pl.len(), n_pos=(pl.col('r') > 0).sum(), q_slice=pl.col('q_slice').first()).group_by('q_slice').agg(cand_median=pl.col('n_cand').median(), pos_median=pl.col('n_pos').median(), no_pos_share=(pl.col('n_pos') == 0).mean()).sort('q_slice')

q_slice,cand_median,pos_median,no_pos_share
str,f64,f64,f64
"""dormant""",20.0,0.0,0.8671
"""head""",103.0,35.0,0.0
"""long_tail""",30.0,1.0,0.173723
"""new""",20.0,0.0,0.55332
"""torso""",76.0,7.0,0.000285


## Candidate sources

In [8]:
flags = [c for c in pairs.columns if c.startswith('cand_')]
pl.DataFrame([{ 'source': f, 'share_of_pairs': pairs[f].mean(), 'pos_rate': pairs.filter(pl.col(f))['r'].gt(0).mean()} for f in flags])

source,share_of_pairs,pos_rate
str,f64,f64
"""cand_opened_next""",0.139539,0.855178
"""cand_twohop""",0.475042,0.094052
"""cand_reverse""",0.105406,0.447754
"""cand_popular""",0.38023,0.000205


## Candidate recall (from build stats)

In [9]:
import json
s = json.load(open('../modeling/build_dataset_stats.json'))['train']
{k: s[k] for k in ('candidate_recall_query_side', 'candidate_recall_item_side')}

{'candidate_recall_query_side': {'new': {'truth_pairs': 80067,
   'recall': 0.3479,
   'recall_w': 0.3527},
  'torso': {'truth_pairs': 396913, 'recall': 0.8566, 'recall_w': 0.9427},
  'long_tail': {'truth_pairs': 107350, 'recall': 0.7043, 'recall_w': 0.7591},
  'dormant': {'truth_pairs': 35954, 'recall': 0.0392, 'recall_w': 0.0059},
  'head': {'truth_pairs': 553737, 'recall': 0.6433, 'recall_w': 0.902}},
 'candidate_recall_item_side': {'torso': {'truth_pairs': 413687,
   'recall': 0.7093,
   'recall_w': 0.7886},
  'new': {'truth_pairs': 8731, 'recall': 0.2434, 'recall_w': 0.2415},
  'head': {'truth_pairs': 604906, 'recall': 0.7044, 'recall_w': 0.8485},
  'long_tail': {'truth_pairs': 140558, 'recall': 0.565, 'recall_w': 0.5354},
  'dormant': {'truth_pairs': 6139, 'recall': 0.0013, 'recall_w': 0.0001}}}